# DPO — Direct Preference Optimization: Your Language Model is Secretly a Reward Model (2023)

_Papers · Transformers & LLMs_

**Align a language model to human preferences with one classification loss — no reward model, no reinforcement learning.**

---

This notebook builds DPO from scratch, one step at a time: the implicit reward, the loss for a single preference pair, a tiny trainable policy, and the training loop that raises preferred responses above dispreferred ones — plus a shuffled-label ablation that shows the signal lives entirely in the labels. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Compute the DPO loss for one preference pair

DPO's implicit reward for a response is `r = beta * (log pi_policy - log pi_ref)` — the scaled log-ratio of the policy against a frozen reference. For a preference pair we want the preferred response's reward above the dispreferred one's, and the loss is `-log sigma(r_w - r_l)`.

Here we plug in the worked example's log-probs and confirm the margin `0.10` and loss `0.6444` match the lesson.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# Sanity-check the worked example: DPO loss for ONE preference pair.
beta_we = 0.1
lp_pol_w, lp_pol_l = -2.0, -3.0      # policy log-probs for (y_w, y_l)
lp_ref_w, lp_ref_l = -2.5, -2.5      # frozen reference log-probs

rw = beta_we * (lp_pol_w - lp_ref_w)  # implicit reward, preferred
rl = beta_we * (lp_pol_l - lp_ref_l)  # implicit reward, dispreferred
margin = rw - rl
loss_we = -np.log(1 / (1 + np.exp(-margin)))

print("worked example: r_w=%.2f r_l=%.2f  margin=%.2f  sigma=%.6f  L_DPO=%.6f" % (
      rw, rl, margin, 1 / (1 + np.exp(-margin)), loss_we))
# worked example: r_w=0.05 r_l=-0.05  margin=0.10  sigma=0.524979  L_DPO=0.644397

### Step 2 — Set up a tiny policy, a frozen reference, and preference pairs

Instead of a full language model, our 'policy' is a categorical distribution over `N` candidate responses: `log pi(y|x) = log_softmax(logits)`. The **reference** policy is random logits that we freeze and never update. The trainable policy is initialized *at* the reference so the implicit reward margin starts exactly at zero.

The `pairs` encode a human ranking — response 0 is best, response 5 worst — as `(y_w, y_l)` 'winner beats loser' pairs.

In [ ]:
# A tiny 'policy': a categorical distribution over N candidate responses.
# log pi(y|x) = log_softmax(logits). (On a real LM this is summed token log-probs.)
N = 6
beta = 0.5                                     # visible temperature for the toy run

# Frozen REFERENCE policy: random logits, then never updated.
torch.manual_seed(1)
ref_logits = torch.randn(N)
ref_logp = F.log_softmax(ref_logits, dim=0).detach()   # log pi_ref(y|x), frozen

# Trainable policy, initialized AT the reference (so the margin starts at 0).
theta = ref_logits.clone().detach().requires_grad_(True)
opt = torch.optim.SGD([theta], lr=0.5)

# Preference pairs (y_w, y_l): a better-ranked response beats a worse one.
# 'Human' ranking: response 0 best ... response 5 worst.
pairs = [(0, 1), (0, 2), (1, 3), (2, 4), (1, 4), (3, 5), (0, 5), (2, 5)]

### Step 3 — Define the DPO loss and train the policy

`dpo_loss` (Eqn. 7) computes, for each pair, the implicit-reward margin `beta * ((logp_w - ref_w) - (logp_l - ref_l))`, then averages `-log sigma` over the pairs. `reward_margin` reports the mean margin on the true pairs so we can watch it grow.

We run plain SGD for 60 steps. As the loss falls, the preferred responses' probabilities should rise and the worst responses' fall toward zero.

In [ ]:
# The NOVEL part, built by hand: the DPO loss (Eqn. 7).
def dpo_loss(logits):
    logp = F.log_softmax(logits, dim=0)        # log pi_theta(y|x)
    z = torch.stack([                          # implicit reward margin per pair
        beta * ((logp[w] - ref_logp[w]) - (logp[l] - ref_logp[l]))
        for (w, l) in pairs])
    return -F.logsigmoid(z).mean()             # -log sigma, averaged over pairs

def reward_margin(logits):                     # mean implicit reward margin over true pairs
    logp = F.log_softmax(logits, dim=0)
    r = beta * (logp - ref_logp)
    return float(np.mean([(r[w] - r[l]).item() for (w, l) in pairs]))

print("\nDPO training (N=%d responses, %d preference pairs, beta=%.1f):" % (N, len(pairs), beta))
for step in range(61):
    if step % 10 == 0:
        print("  step %2d  loss=%.4f  mean reward-margin=%.4f" % (
              step, dpo_loss(theta).item(), reward_margin(theta)))
    opt.zero_grad()
    dpo_loss(theta).backward()
    opt.step()

# Did the PREFERRED responses' probabilities rise?
with torch.no_grad():
    p_init = torch.softmax(ref_logits, dim=0)
    p_fin = torch.softmax(theta, dim=0)
print("\nresponse probs  init :", [round(v, 3) for v in p_init.tolist()])
print("response probs  final:", [round(v, 3) for v in p_fin.tolist()])
# response 0 (most preferred) rises ~0.25 -> ~0.69; the worst responses fall toward 0.

### Step 4 — Ablation: shuffle the preference labels

DPO has no built-in notion of quality — it only enforces the orderings you label. To prove that, we randomly flip the winner/loser in some pairs and retrain from the reference, everything else identical.

With contradictory supervision the gradients fight each other, so the margin measured on the *true* pairs barely moves — confirming the signal lives entirely in the labels.

In [ ]:
# ABLATION: shuffle preference labels -> the true-pair margin stops growing.
torch.manual_seed(2)
theta2 = ref_logits.clone().detach().requires_grad_(True)
opt2 = torch.optim.SGD([theta2], lr=0.5)

# Randomly flip (y_w, y_l) in roughly half the pairs.
bad_pairs = [(l, w) if random.random() < 0.5 else (w, l) for (w, l) in pairs]

def dpo_loss_bad(logits):
    logp = F.log_softmax(logits, dim=0)
    z = torch.stack([
        beta * ((logp[w] - ref_logp[w]) - (logp[l] - ref_logp[l]))
        for (w, l) in bad_pairs])
    return -F.logsigmoid(z).mean()

for step in range(60):
    opt2.zero_grad()
    dpo_loss_bad(theta2).backward()
    opt2.step()

print("\nablation (shuffled labels): TRUE-pair mean reward-margin = %.4f" % reward_margin(theta2))

# Typical output (our small run, not the paper's numbers; exact values vary by seed/hardware):
#   step  0  loss=0.6931  mean reward-margin=0.0000
#   step 60  loss=0.3348  mean reward-margin=0.9827
#   response probs  init : [0.253, 0.171, 0.139, 0.243, 0.083, 0.111]
#   response probs  final: [0.693, 0.128, 0.088, 0.077, 0.008, 0.006]
#   ablation (shuffled labels): TRUE-pair mean reward-margin = 0.1143

## Visualize the data & results

_As we train a tiny policy with the DPO loss on preference pairs, does the implicit reward margin between preferred and dispreferred responses grow — and does training on SHUFFLED labels?_

### Step 1 — Rebuild the policy, reference, and pairs for the plot

This visualization cell is self-contained, so it re-imports and re-creates the same frozen reference and preference pairs as the reference implementation. The `margin` helper measures the mean implicit reward margin on the true pairs at any point in training.

Keeping the setup identical means the two curves we plot are directly comparable.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

N, beta = 6, 0.5
torch.manual_seed(1)
ref_logits = torch.randn(N)
ref_logp = F.log_softmax(ref_logits, dim=0).detach()      # frozen reference
pairs = [(0, 1), (0, 2), (1, 3), (2, 4), (1, 4), (3, 5), (0, 5), (2, 5)]  # y_w beats y_l

def margin(logits):
    logp = F.log_softmax(logits, dim=0)
    r = beta * (logp - ref_logp)
    return float(np.mean([(r[w] - r[l]).item() for (w, l) in pairs]))

### Step 2 — Train on correct vs shuffled labels and compare

`train` runs DPO from the reference on whatever pairs you pass, recording the true-pair margin every 10 steps. We run it twice: once on the correct pairs, once on a randomly-shuffled copy — and always measure the margin on the *true* pairs.

The correct-label margin should climb toward ~0.98 while the shuffled-label margin stays near ~0.11, the same contrast as the ablation above.

In [ ]:
def train(use_pairs):
    th = ref_logits.clone().detach().requires_grad_(True)   # init AT reference
    opt = torch.optim.SGD([th], lr=0.5)
    curve = {0: margin(th)}
    for step in range(1, 61):
        logp = F.log_softmax(th, dim=0)
        z = torch.stack([
            beta * ((logp[w] - ref_logp[w]) - (logp[l] - ref_logp[l]))
            for (w, l) in use_pairs])
        loss = -F.logsigmoid(z).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % 10 == 0:
            curve[step] = margin(th)        # always measured on TRUE pairs
    return curve

correct = train(pairs)
bad = [(l, w) if random.random() < 0.5 else (w, l) for (w, l) in pairs]
shuffled = train(bad)

xs = [0, 10, 20, 30, 40, 50, 60]
print("correct  margin:", [round(correct[s], 4) for s in xs])
print("shuffled margin:", [round(shuffled[s], 4) for s in xs])
# correct  margin: [0.0, 0.2199, 0.4112, 0.5791, 0.7281, 0.8617, 0.9827]
# shuffled margin: [0.0, 0.0195, 0.0389, 0.0581, 0.0771, 0.0959, 0.1143]
# Correct labels: margin grows to ~0.98. Shuffled: ~0.11. Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The shuffled-label ablation. You have DPO working: the implicit reward margin on the true
            preference pairs grows during training. Now shuffle the labels &mdash; randomly swap $y_w$
            and $y_l$ in some pairs &mdash; and retrain, everything else identical. What do you expect to
            happen to the reward margin measured on the true (correct) pairs, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what the loss optimizes: $-\log\sigma(\hat r(y_w)-\hat r(y_l))$ pushes the margin positive for whatever pair you label as $(y_w,y_l)$. — _The loss only knows the labels you give it; it has no independent notion of "correct"._
- Shuffle: now roughly half the pairs point the right way and half the wrong way, so the gradients fight each other. — _Up-pushing some pairs and down-pushing their reverses roughly cancels, leaving no consistent direction._
- Measure the margin on the TRUE pairs: expect it to stay near zero, not grow. — _With contradictory supervision there is no signal to separate truly-preferred from truly-dispreferred responses._

**Answer:** The reward margin on the true pairs stops growing &mdash; it stays near zero. DPO has no
                 built-in notion of quality; it only enforces the orderings you label. Shuffling the labels gives
                 contradictory supervision, so the pushes cancel and the policy stays close to the reference. In
                 our run, training on shuffled labels leaves the true-pair mean reward margin at $\approx 0.11$,
                 versus $\approx 0.98$ after training on the correct labels. The signal lives entirely in the
                 preference labels.

</details>

**Problem 2.** Why does DPO need no reward model and no reinforcement learning, when RLHF needs both? Point
            to the exact step in the derivation that removes each.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reward model: in RLHF you fit $r$ from preferences, then optimize against it. In DPO, Eqn. 5 writes $r(x,y)=\beta\log\frac{\pi}{\pi_{\text{ref}}}+\beta\log Z(x)$ &mdash; the reward is a function of the policy. — _If the reward is already determined by the policy, fitting a separate reward network is redundant._
- Reinforcement learning: RLHF samples from the policy because the optimal policy (Eqn. 4) contains the intractable $Z(x)$. DPO sidesteps it because $Z(x)$ cancels in the preference difference (Eqn. 6). — _No intractable normalizer means no sampling loop &mdash; the objective becomes a plain supervised loss._
- Conclude: substituting Eqn. 5 into Bradley-Terry collapses the whole pipeline to one classification loss (Eqn. 7). — _Both the reward-fitting and the RL stages disappear into a single maximum-likelihood objective._

**Answer:** No reward model because Eqn. 5 expresses the reward directly as the scaled log-ratio
                 $\beta\log\frac{\pi_\theta}{\pi_{\text{ref}}}$ &mdash; the policy is the reward
                 model. No reinforcement learning because the intractable partition function $Z(x)$, which
                 forced RLHF to sample, cancels in the Bradley-Terry preference difference (Eqn. 6). What
                 remains (Eqn. 7) is an ordinary logistic-regression loss over preference pairs &mdash; one
                 supervised objective replacing the entire reward-fit-then-PPO pipeline.

</details>

**Problem 3.** In the worked example you had $\beta=0.1$, policy log-probs $(-2.0, -3.0)$ for $(y_w, y_l)$ and
            reference log-probs $(-2.5, -2.5)$, giving margin $0.10$ and loss $0.6444$. Suppose training raises
            the policy's preferred log-prob to $\log\pi_\theta(y_w|x)=-1.0$ (dispreferred and reference
            unchanged). Recompute the margin and the loss. What does it show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- New implicit reward, preferred: $\hat r(y_w)=0.1\cdot(-1.0-(-2.5))=0.1\cdot 1.5 = 0.15$. — _Raising the policy's probability on $y_w$ raises its implicit reward, since reference is fixed._
- Dispreferred unchanged: $\hat r(y_l)=0.1\cdot(-3.0-(-2.5))=-0.05$. New margin $z=0.15-(-0.05)=0.20$. — _Only the preferred side moved, so the margin grows from $0.10$ to $0.20$._
- New loss: $-\log\sigma(0.20)=-\log(0.549834)=0.598139$. — _A larger positive margin means a higher $\sigma$, hence a lower $-\log\sigma$ loss._

**Answer:** The margin rises from $0.10$ to $\mathbf{0.20}$ and the loss falls from $0.6444$ to
                 $\approx\mathbf{0.5981}$. Raising the policy's probability on the preferred response (relative
                 to the frozen reference) increases its implicit reward, widens the margin, and lowers the DPO
                 loss &mdash; exactly the gradient direction DPO follows. This is the mechanism by which DPO
                 "raises the probability of preferred responses above dispreferred ones."

</details>